# SE-LLM-350M — Kaggle SFT (Instruction Fine-Tuning) Notebook

**Run AFTER pre-training is complete.**

This notebook fine-tunes the pre-trained base model on 255K instruction-response pairs
so the model learns to follow natural language coding instructions.

**Instructions:**
1. Enable GPU: Settings → Accelerator → P100
2. Add your dataset: `se-llm-data` (containing sft_data.jsonl + tokenizer.json)
3. Add your pre-trained checkpoint: `se-llm-checkpoints` dataset
4. Add Kaggle Secret: `WANDB_API_KEY`, `GITHUB_TOKEN`
5. Click **Run All**

> SFT takes ~4-6 hours — fits in a single Kaggle session.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU required! Enable P100 in Settings → Accelerator'

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q wandb tokenizers datasets pyyaml

In [ ]:
# ── Cell 3: Clone training code ───────────────────────────────
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_REPO = 'YOUR_GITHUB_USERNAME/se-llm-350m'  # ← UPDATE THIS

if not os.path.exists('/kaggle/working/se-llm-350m'):
    try:
        token    = secrets.get_secret('GITHUB_TOKEN')
        repo_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
    except Exception:
        repo_url = f'https://github.com/{GITHUB_REPO}.git'
    !git clone {repo_url} /kaggle/working/se-llm-350m
else:
    !git -C /kaggle/working/se-llm-350m pull

%cd /kaggle/working/se-llm-350m
!ls -la

In [ ]:
# ── Cell 4: Link dataset files ────────────────────────────────
import os

os.makedirs('data/sft', exist_ok=True)
os.makedirs('tokenizer', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('checkpoints_sft', exist_ok=True)

DATASET_PATH = '/kaggle/input/se-llm-data'

# Link SFT instruction dataset
sft_src  = f'{DATASET_PATH}/sft_data.jsonl'
sft_dest = 'data/sft/sft_data.jsonl'
if os.path.exists(sft_src) and not os.path.exists(sft_dest):
    os.symlink(sft_src, sft_dest)
    print(f'Linked SFT data')

# Link tokenizer
tok_src  = f'{DATASET_PATH}/tokenizer.json'
tok_dest = 'tokenizer/tokenizer.json'
if os.path.exists(tok_src) and not os.path.exists(tok_dest):
    os.symlink(tok_src, tok_dest)
    print(f'Linked tokenizer')

# Show SFT data info
if os.path.exists(sft_dest):
    with open(sft_dest) as f:
        n_lines = sum(1 for _ in f)
    size_mb = os.path.getsize(sft_dest) / 1e6
    print(f'SFT dataset: {n_lines:,} samples | {size_mb:.1f} MB')
else:
    print('WARNING: sft_data.jsonl not found — check dataset name')

In [ ]:
# ── Cell 5: Load pre-trained base model checkpoint ────────────
import os, shutil, glob

# The pre-trained checkpoint from kaggle_pretrain.ipynb output
PRETRAIN_CKPT_PATH = '/kaggle/input/se-llm-checkpoints'

best_ckpt = os.path.join(PRETRAIN_CKPT_PATH, 'best.pt')
latest_ckpt = os.path.join(PRETRAIN_CKPT_PATH, 'latest.pt')

base_checkpoint = None

if os.path.exists(best_ckpt):
    shutil.copy(best_ckpt, 'checkpoints/pretrain_best.pt')
    base_checkpoint = 'checkpoints/pretrain_best.pt'
    print(f'Loaded pre-training best checkpoint')
elif os.path.exists(latest_ckpt):
    shutil.copy(latest_ckpt, 'checkpoints/pretrain_latest.pt')
    base_checkpoint = 'checkpoints/pretrain_latest.pt'
    print(f'Loaded pre-training latest checkpoint')
else:
    # Check for any .pt file
    pt_files = glob.glob(f'{PRETRAIN_CKPT_PATH}/*.pt')
    if pt_files:
        src = sorted(pt_files)[-1]
        shutil.copy(src, 'checkpoints/pretrain_ckpt.pt')
        base_checkpoint = 'checkpoints/pretrain_ckpt.pt'
        print(f'Loaded checkpoint: {os.path.basename(src)}')
    else:
        print('WARNING: No pre-trained checkpoint found!')
        print('Make sure you added the se-llm-checkpoints dataset from the pre-training output')

if base_checkpoint:
    import torch
    ckpt = torch.load(base_checkpoint, map_location='cpu', weights_only=False)
    print(f'  Step:   {ckpt.get("step", "unknown"):,}')
    print(f'  Tokens: {ckpt.get("tokens_processed", 0)/1e9:.3f}B')
    print(f'  Loss:   {ckpt.get("val_loss", 0):.4f}')

In [ ]:
# ── Cell 6: Login to W&B ──────────────────────────────────────
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets   = UserSecretsClient()
    wandb_key = secrets.get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('W&B logged in')
except Exception as e:
    print(f'W&B login skipped: {e}')

In [ ]:
# ── Cell 7: RUN INSTRUCTION FINE-TUNING ───────────────────────
# This runs SFT for ~4-6 hours on Kaggle P100.
# It takes the pre-trained base model and teaches it
# to follow natural language software engineering instructions.

import subprocess, sys

base_ckpt_arg = f'--base-checkpoint {base_checkpoint}' if base_checkpoint else ''

cmd = f'python training/sft.py --config configs/350m.yaml {base_ckpt_arg}'
print(f'Running: {cmd}\n')
!{cmd}

In [ ]:
# ── Cell 8: Quick quality test ─────────────────────────────────
# Test the SFT model with a few sample prompts

import torch, sys
sys.path.insert(0, '/kaggle/working/se-llm-350m')

from evaluation.generate import load_model_from_checkpoint, load_tokenizer, chat_turn

device    = torch.device('cuda')
ckpt_path = 'checkpoints_sft/latest.pt'

if not os.path.exists(ckpt_path):
    print('SFT checkpoint not found — check training completed successfully')
else:
    model, cfg = load_model_from_checkpoint(ckpt_path, device)
    tokenizer  = load_tokenizer('tokenizer/tokenizer.json')

    test_prompts = [
        'Write a Python function to check if a number is prime.',
        'Write a SQL query to find the top 3 most expensive products.',
        'Convert this Python list to JavaScript: [1, 2, 3, 4, 5]',
    ]

    for prompt in test_prompts:
        print(f'\n{"─"*50}')
        print(f'User: {prompt}')
        response = chat_turn(model, tokenizer, prompt, device=device, max_new_tokens=200)
        print(f'SE-LLM:\n{response}')

    print('\n✅ Model quality check complete')

In [ ]:
# ── Cell 9: Save SFT checkpoint to output ─────────────────────
import shutil, glob, os

output_dir = '/kaggle/working/output'
os.makedirs(output_dir, exist_ok=True)

# Copy all SFT checkpoints to output
for ckpt in glob.glob('checkpoints_sft/*.pt'):
    dest = f'{output_dir}/{os.path.basename(ckpt)}'
    shutil.copy(ckpt, dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f'Saved: {dest} ({size_mb:.0f} MB)')

print('\n✅ SFT complete! Download the output files.')
print('Next step: run evaluation/humaneval.py on the SFT model')